## Tabela: `workspace.silver.municipio`

| Item | Detalhe |
|------|---------|
| **Tabela** | `workspace.silver.municipio` |
| **Origem** | `workspace.bronze.municipio_raw` |
| **Descrição** | Tratamento e padronização dos dados de municípios, previamente ingeridos na camada Bronze. |
| **Camada** | Silver (tratamento e padronização) |
| **Colunas de controle** | `data_ingestao` (timestamp de carga), `fonte` (origem dos dados) |
| **Criado por** | suelen.ramos |
| **Data de criação** | 2025-01-20 |

### Chave de negócio
A combinação `ano + id_municipio + serie + rede_codigo` forma a chave única da tabela.

### Tratamentos realizados

- Padronização dos tipos de dados;
- Normalização das chaves;
- Padronização dos nomes das colunas;
- Preservação dos indicadores originais;
- Preservação das colunas de controle da ingestão;
- Validação de duplicidades após a transformação;
- Validação de valores nulos.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.silver.municipio
USING DELTA
AS
SELECT
    CAST(ano AS INT) AS ano,
    TRIM(CAST(id_municipio AS STRING)) AS id_municipio,
    TRIM(CAST(serie AS STRING)) AS serie,
    TRIM(CAST(rede AS STRING)) AS rede_codigo,

    CAST(taxa_alfabetizacao AS DOUBLE) AS taxa_alfabetizacao,
    CAST(media_portugues AS DOUBLE) AS media_portugues,

    CAST(proporcao_aluno_nivel_0 AS DOUBLE) AS proporcao_aluno_nivel_0,
    CAST(proporcao_aluno_nivel_1 AS DOUBLE) AS proporcao_aluno_nivel_1,
    CAST(proporcao_aluno_nivel_2 AS DOUBLE) AS proporcao_aluno_nivel_2,
    CAST(proporcao_aluno_nivel_3 AS DOUBLE) AS proporcao_aluno_nivel_3,
    CAST(proporcao_aluno_nivel_4 AS DOUBLE) AS proporcao_aluno_nivel_4,
    CAST(proporcao_aluno_nivel_5 AS DOUBLE) AS proporcao_aluno_nivel_5,
    CAST(proporcao_aluno_nivel_6 AS DOUBLE) AS proporcao_aluno_nivel_6,
    CAST(proporcao_aluno_nivel_7 AS DOUBLE) AS proporcao_aluno_nivel_7,
    CAST(proporcao_aluno_nivel_8 AS DOUBLE) AS proporcao_aluno_nivel_8,

    data_ingestao,
    fonte,
    lote_id

FROM workspace.bronze.municipio_raw

WHERE ano IS NOT NULL
  AND id_municipio IS NOT NULL
  AND serie IS NOT NULL
  AND rede IS NOT NULL

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ano, id_municipio, serie, rede
    ORDER BY data_ingestao DESC
) = 1
""")

print("Tratamento dos municípios concluído na Silver!")

In [0]:
registrar_execucao("SILVER", "municipio", _m, row_count=contar_tabela("workspace.silver.municipio"))